# 03 — Sliding-window LSTM Sequence Model

Train a stacked LSTM on **raw normalized sensor sequences** to predict `% Iron Concentrate` and `% Silica Concentrate` jointly. Compare against the LightGBM baseline from notebook 02 on the same temporal split.

**Methodological choices:**

1. **Z-score** each sensor using **train-only** statistics (no leakage).
2. **Sliding window of 180 cycles** (1 hour at 20-second sampling) ending at every cycle of the train/val/test sets.
3. **Multi-output regression**: a single LSTM with two output heads (one per target).
4. **Early stopping** on validation RMSE with patience 6.
5. **MLflow tracking** with model signature.

**Hypothesis:** the LSTM should match or beat LightGBM when sequence context (process delay, ramp-ups, oscillations) carries information beyond what static rolling features capture. Honest expectation: similar performance on both, with LSTM possibly winning on `% Silica Concentrate` (smaller signal, more sensitive to short-term dynamics).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from frothiq.data.loader import SENSOR_COLS, FEED_COLS, TARGET_COLS, load_flotation, temporal_split
from frothiq.models.deep import train_lstm

mlflow.set_tracking_uri(f'file:{(ROOT / "mlruns").as_posix()}')
mlflow.set_experiment('frothiq-lstm')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | device = {device}')

PyTorch 2.11.0+cpu | device = cpu


## 1. Load raw flotation data + temporal split

In [2]:
data = load_flotation()
df = data.df

# Use sensor + feed columns as inputs to the LSTM (raw, not engineered).
input_cols = data.sensor_cols + data.feed_cols
print(f'Input columns: {len(input_cols)}')
print(f'Target columns: {TARGET_COLS}')

train, val, test = temporal_split(df, train_frac=0.7, val_frac=0.15)
# Drop rows with NaN in any input or target.
train = train.dropna(subset=input_cols + TARGET_COLS).reset_index(drop=True)
val = val.dropna(subset=input_cols + TARGET_COLS).reset_index(drop=True)
test = test.dropna(subset=input_cols + TARGET_COLS).reset_index(drop=True)
print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')

Input columns: 21
Target columns: ['pct_iron_concentrate', 'pct_silica_concentrate']
Train: 516,217  Val: 110,618  Test: 110,618


## 2. Train LSTM

On a CPU this takes a few minutes for 30 epochs over ~500K windows. With a GPU it's seconds.

In [3]:
t0 = time.time()
result = train_lstm(
    train_df=train,
    val_df=val,
    sensor_cols=input_cols,
    target_cols=TARGET_COLS,
    window=180,            # 1 hour at 20-second sampling
    hidden_size=64,
    num_layers=2,
    dropout=0.25,
    lr=1e-3,
    batch_size=256,
    max_epochs=30,
    patience=6,
    test_df=test,
    device=device,
    run_name='lstm-multi-target',
)
elapsed = time.time() - t0
print(f'\nTrained in {elapsed:.1f}s | best epoch = {result.best_epoch}')
print(f'Validation: {result.val_metrics}')
print(f'Test:       {result.test_metrics}')

KeyboardInterrupt: 

## 3. Compare against LightGBM (from notebook 02)

If you've run notebook 02, you have `lgbm-pct_iron_concentrate` and `lgbm-pct_silica_concentrate` runs in MLflow. Pull their test metrics for comparison.

If you haven't run notebook 02 yet, plug your LightGBM numbers into `lgbm_test_rmse` below for the comparison plot.

In [ ]:
# If notebook 02 was executed, this query pulls the latest LightGBM run per target.
# Adjust experiment_name if you used a different one.
lgbm_runs = mlflow.search_runs(experiment_names=['frothiq-baseline'])
if len(lgbm_runs):
    print(f'Found {len(lgbm_runs)} LightGBM runs.')
    print(lgbm_runs[['tags.mlflow.runName', 'metrics.test_rmse', 'metrics.test_r2']].head())
else:
    print('No prior baseline runs found. Run notebook 02 first to populate the comparison.')

## 4. Predicted vs true on test set

Visual sanity check: scatter of predictions against ground truth for both targets.

In [ ]:
from frothiq.models.deep.lstm import make_windows

X_test, y_test = make_windows(test, input_cols, window=180, target_cols=TARGET_COLS, scaler=result.scaler)
result.model.eval()
with torch.no_grad():
    y_pred = result.model(torch.from_numpy(X_test).to(device)).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, i, target in zip(axes, range(2), TARGET_COLS):
    ax.scatter(y_test[:, i], y_pred[:, i], alpha=0.05, s=4)
    lims = [min(y_test[:, i].min(), y_pred[:, i].min()), max(y_test[:, i].max(), y_pred[:, i].max())]
    ax.plot(lims, lims, 'r--', alpha=0.6, label='ideal')
    ax.set_xlabel(f'true {target}')
    ax.set_ylabel(f'predicted {target}')
    ax.set_title(f'{target} (LSTM)')
    ax.legend()
fig.tight_layout()

## 5. Honest take

Once you compare against LightGBM, two outcomes are likely:

* **LightGBM ties or wins**: the rolling/lag features in notebook 01 already capture the temporal dynamics. The LSTM doesn't add value beyond a tabular model on engineered features. *Reported honestly, this is information about the dataset, not a bug.*
* **LSTM wins**: the sequence captures dynamics the static features missed (oscillations, ramps). Use it in production for that target.

Either way: the comparison itself is the value. We don't pre-decide which model to use; we measure on the test set with the same temporal split.

## What's next: `04_spc.ipynb`

Apply Shewhart, CUSUM, and EWMA charts to the residuals of the best model. Detect when predictions start drifting before lab measurements catch up.